# **Sesión14: Limpieza y Transformación de Datos: Datos Duplicados**
###Detección y eliminación con `duplicated()` y `drop_duplicates()`

---

| | |
|---|---|
| **Materia** | Programación para Analítica Descriptiva y Predictiva |
| **Programa** | Maestría en Inteligencia Artificial y Analítica de Datos (MIAAD) — UACJ |
| **Unidad** | 02 Análisis Descriptivo de los Datos |
| **Tema** | Limpieza y transformación de datos: duplicados |
| **Entorno** | Google Colab |

---

## Objetivo

Comprender qué son los datos duplicados, por qué distorsionan el análisis y cómo **detectarlos**, **cuantificarlos** y **tratarlos** en un `DataFrame` de pandas mediante `duplicated()` y `drop_duplicates()`.

## Agenda

| # | Sección | Contenido |
|---|---|---|
| 1 | Introducción | Qué es un duplicado, tipos y efectos en el análisis |
| 2 | `duplicated()` | Sintaxis, parámetro `keep` |
| 3 | Duplicidad parcial | Parámetro `subset` |
| 4 | Cuantificar | `.sum()`, proporción, `value_counts()` |
| 5 | Filtrar | Máscaras booleanas y el operador `~` |
| 6 | `drop_duplicates()` | Sintaxis, `keep`, `subset`, `ignore_index`, `inplace` |
| 7 | Casos avanzados | Integración con `pd.concat()`, duplicados ocultos, `NaN`, tipos, registro más reciente, registro más completo, consolidación con `groupby()`, conflictos |
| 8 | Dataset real | Customer Personality Analysis |
| 9 | Cierre | Resumen y buenas prácticas |

## Cómo usar este notebook

Ejecuta las celdas en orden (`Shift + Enter`). Antes de algunas celdas encontrarás la indicación 🔮 **Predicción**: piensa qué resultado esperas, ejecuta la celda y compara con lo que obtuviste. La celda de **Observación** que sigue explica el resultado.

Las celdas marcadas con ⚠️ producen un **error intencional** para mostrar fallas frecuentes; cada una va seguida de su explicación.

## 1. Introducción a los datos duplicados

### 1.1 ¿Qué es un dato duplicado?

Un **dato duplicado** es un registro (fila) que aparece más de una vez en un conjunto de datos. Suelen originarse por:

- Errores de captura (un formulario enviado dos veces).
- Integración de fuentes (dos sistemas que registran al mismo cliente).
- Procesos de carga repetidos (un script que agrega el mismo archivo dos veces).

### 1.2 Tipos de duplicidad

| Tipo | Definición | Ejemplo |
|---|---|---|
| **Exacta** | Las filas coinciden en **todas** sus columnas. | `[1, Ana, 25]` aparece dos veces. |
| **Parcial** | Las filas coinciden solo en **algunas** columnas elegidas (columnas clave). | Dos filas con `ID = 1`, una con `Ana` y otra con `Vicente`. |
| **Oculta** | Representan lo mismo, pero el texto difiere en formato (mayúsculas, espacios). | `"Ana López"` y `"ana lópez "`. |

Considera la siguiente tabla:

| ID | Nombre | Edad |
|---|---|---|
| 1 | Ana | 25 |
| 2 | Juan | 30 |
| 3 | Luis | 35 |
| 4 | Sara | 40 |
| 1 | Ana | 25 |
| 2 | Juan | 30 |
| 2 | Juan | 30 |
| 1 | Vicente | 48 |
| 3 | Rogelio | 40 |

- **Duplicidad exacta**: `[1, Ana, 25]` y `[2, Juan, 30]`.
- **Duplicidad parcial**: depende de las columnas elegidas. En `ID` se repiten 1, 2 y 3; en `Nombre`, Ana y Juan; en `Edad`, 25, 30 y 40.

Observa que `[1, Vicente, 48]` comparte el `ID` con Ana, pero **no** es la misma persona. Un duplicado parcial no siempre es un error que deba eliminarse: puede ser un **conflicto** que requiere revisión (lo veremos en la sección 7).

### 1.3 ¿Por qué importan? Efectos en el análisis

1. **Distorsión de métricas**: inflan conteos, promedios y frecuencias.
2. **Errores en modelos predictivos**: el modelo "ve" varias veces el mismo ejemplo y le da más peso; esto favorece el **sobreajuste** (*overfitting*). Si un duplicado queda en entrenamiento y su copia en prueba, la evaluación del modelo resulta optimista (fuga de datos o *data leakage*).
3. **Falsas correlaciones**: las relaciones entre variables parecen más fuertes de lo que son.
4. **Costo computacional**: más memoria y más tiempo de procesamiento.

> En este notebook, dos registros son **idénticos** cuando sus **valores** coinciden. En modelos predictivos la similitud puede definirse de otra forma (por ejemplo, con una métrica de distancia), pero eso queda fuera del alcance de esta sesión.

### 1.4 Ejemplo: efecto de los duplicados en la media

Construiremos un `DataFrame` de empleados con varios registros repetidos y calcularemos la media del salario.

**Pregunta**: si una persona con salario alto aparece más veces que las demás, ¿la media sube o baja?

In [ ]:
import pandas as pd

# Diccionario con datos de empleados; varios registros están repetidos
data = {
    'ID':      [1, 2, 3, 4, 1, 2, 2, 3, 5, 6, 7, 3, 5, 7, 1, 2],
    'Nombre':  ['Ana', 'Juan', 'Luis', 'Sara', 'Ana', 'Juan', 'Juan', 'Luis',
                'Carlos', 'Elena', 'Mario', 'Luis', 'Carlos', 'Mario', 'Ana', 'Juan'],
    'Edad':    [25, 30, 35, 40, 25, 30, 30, 35, 28, 32, 29, 35, 28, 29, 25, 30],
    'Salario': [50000, 60000, 55000, 70000, 50000, 60000, 60000, 55000,
                48000, 51000, 53000, 55000, 48000, 53000, 50000, 60000]
}

# Convertir el diccionario en DataFrame
df = pd.DataFrame(data)

# Mostrar el DataFrame completo
print(df)

# Media del salario considerando TODAS las filas (incluidos duplicados)
print('\nNúmero de filas:', len(df))
print('Media del salario CON duplicados:', df['Salario'].mean())

Ahora construimos manualmente la versión sin duplicados: un registro por empleado.

In [ ]:
# Misma información, pero cada empleado aparece una sola vez
data_unicos = {
    'ID':      [1, 2, 3, 4, 5, 6, 7],
    'Nombre':  ['Ana', 'Juan', 'Luis', 'Sara', 'Carlos', 'Elena', 'Mario'],
    'Edad':    [25, 30, 35, 40, 28, 32, 29],
    'Salario': [50000, 60000, 55000, 70000, 48000, 51000, 53000]
}

df_unicos = pd.DataFrame(data_unicos)
print(df_unicos)

print('\nNúmero de filas:', len(df_unicos))
print('Media del salario SIN duplicados:', df_unicos['Salario'].mean())

**Observación**: la media cambia de **54,875.00** a **55,285.71**. Sara (salario más alto, 70,000) aparece una sola vez, mientras que Juan aparece cuatro veces; los duplicados le dan a Juan más peso del que le corresponde.

Construir la tabla limpia a mano no es viable con miles de filas. Pandas ofrece dos métodos para hacerlo de forma automática:

| Método | ¿Qué hace? | ¿Qué devuelve? |
|---|---|---|
| `df.duplicated()` | **Detecta** filas repetidas | Una `Series` booleana (`True`/`False`) |
| `df.drop_duplicates()` | **Elimina** filas repetidas | Un nuevo `DataFrame` |

## 2. Detección con `duplicated()`

### Método: `DataFrame.duplicated()`

```python
df.duplicated(subset=None, keep='first')
```

Devuelve una `Series` booleana con una posición por fila:

- `True`: la fila es un duplicado (sus valores ya aparecieron en otra fila).
- `False`: la fila no está repetida, o es la ocurrencia que se "conserva".

| Parámetro | Valores | Significado |
|---|---|---|
| `subset` | `None` (por defecto) o lista de columnas | Columnas que se comparan. `None` compara todas (duplicidad exacta). |
| `keep` | `'first'` (por defecto) | Marca como `True` todas las repeticiones **excepto la primera**. |
| | `'last'` | Marca como `True` todas las repeticiones **excepto la última**. |
| | `False` (booleano, sin comillas) | Marca como `True` **todas** las ocurrencias, incluida la primera. |

> `duplicated()` compara **valores de las columnas**, no el índice. Dos filas con índices distintos pero valores iguales se consideran duplicadas.

### 2.1 Duplicidad exacta con `keep='first'` (valor por defecto)

**Pregunta**: la fila 0 es `[1, Ana, 25, 50000]` y vuelve a aparecer en las filas 4 y 14. ¿Cuáles de esas tres filas quedarán marcadas como `True`?

In [ ]:
# Sin argumentos: compara todas las columnas y conserva la primera aparición
print(df.duplicated())

**Observación**: la fila 0 es `False` porque es la **primera** aparición de Ana; las filas 4 y 14 son `True`. Lo mismo ocurre con Juan (fila 1 → `False`; filas 5, 6 y 15 → `True`).

Leer una columna de `True`/`False` separada del `DataFrame` es incómodo. Para comparar los tres valores de `keep` lado a lado, agregamos cada resultado como columna a una **copia** del `DataFrame`.

In [ ]:
# Hacemos una copia para no alterar el DataFrame original
comparacion = df.copy()

# Cada columna nueva guarda el resultado de duplicated() con un valor distinto de keep
comparacion['keep_first'] = df.duplicated(keep='first')
comparacion['keep_last']  = df.duplicated(keep='last')
comparacion['keep_False'] = df.duplicated(keep=False)

print(comparacion)

**Observación**: sigue a Ana (filas 0, 4 y 14):

| Fila | `keep='first'` | `keep='last'` | `keep=False` |
|---|---|---|---|
| 0 | False (primera, se conserva) | True | True |
| 4 | True | True | True |
| 14 | True | False (última, se conserva) | True |

Sara (fila 3) y Elena (fila 9) aparecen una sola vez: son `False` en las tres columnas.

**Regla práctica**:
- `keep='first'` o `keep='last'` responden: *¿qué filas sobran?*
- `keep=False` responde: *¿qué filas pertenecen a un grupo repetido?*

## 3. Duplicidad parcial: parámetro `subset`

En la práctica, dos registros pueden representar la misma entidad aunque no coincidan en **todas** las columnas (por ejemplo, un cliente con el mismo correo pero con distinta fecha de registro). El parámetro `subset` indica **qué columnas** se comparan:

```python
df.duplicated(subset=['columna1', 'columna2'], keep='first')
```

- Con **una** columna: se compara solo esa columna.
- Con **varias** columnas: las filas son duplicadas si coinciden **en todas las columnas indicadas a la vez** (no en cualquiera de ellas).

**Pregunta**: en `df`, ¿habrá más duplicados considerando solo `Edad` o considerando todas las columnas?

In [ ]:
# Duplicados considerando únicamente la columna ID
print('Duplicados en ID (keep=first):')
print(df.duplicated(subset=['ID']).sum(), 'filas')

# Duplicados considerando únicamente la columna Edad
print('\nDuplicados en Edad (keep=first):')
print(df.duplicated(subset=['Edad']).sum(), 'filas')

# Duplicados considerando la combinación Nombre + Edad
print('\nDuplicados en Nombre + Edad (keep=first):')
print(df.duplicated(subset=['Nombre', 'Edad']).sum(), 'filas')

# Duplicados considerando todas las columnas (subset=None)
print('\nDuplicados exactos (todas las columnas):')
print(df.duplicated().sum(), 'filas')

**Observación**: en este `DataFrame` los cuatro conteos coinciden (9) porque cada `ID` siempre va acompañado del mismo nombre, edad y salario. Esto **no** es lo habitual. Veamos un caso en el que `subset` sí marca la diferencia.

In [ ]:
# Tabla de ejemplo de la introducción: el ID 1 lo tienen Ana y Vicente,
# el ID 3 lo tienen Luis y Rogelio
personas = pd.DataFrame({
    'ID':     [1, 2, 3, 4, 1, 2, 2, 1, 3],
    'Nombre': ['Ana', 'Juan', 'Luis', 'Sara', 'Ana', 'Juan', 'Juan', 'Vicente', 'Rogelio'],
    'Edad':   [25, 30, 35, 40, 25, 30, 30, 48, 40]
})

comparacion_p = personas.copy()
comparacion_p['exacto']      = personas.duplicated()                            # todas las columnas
comparacion_p['por_ID']      = personas.duplicated(subset=['ID'])               # solo ID
comparacion_p['por_Edad']    = personas.duplicated(subset=['Edad'])             # solo Edad
comparacion_p['ID_y_Nombre'] = personas.duplicated(subset=['ID', 'Nombre'])     # ID y Nombre juntos

print(comparacion_p)
print('\nTotales:')
print(comparacion_p[['exacto', 'por_ID', 'por_Edad', 'ID_y_Nombre']].sum())

**Observación**:

- `exacto` marca 3 filas (las repeticiones de Ana y de Juan).
- `por_ID` marca 5: además incluye a **Vicente** (ID 1) y **Rogelio** (ID 3), que no son la misma persona que Ana y Luis.
- `por_Edad` marca 4: las 3 repeticiones de Ana y Juan, más **Rogelio**, que tiene 40 años igual que Sara; una coincidencia accidental.
- `ID_y_Nombre` vuelve a 3: exigir que coincidan **ambas** columnas es más estricto.

La elección de `subset` es una **decisión de negocio**: define qué significa "el mismo registro". Una sola columna poco informativa (como `Edad`) casi siempre produce falsos duplicados.

## 4. Cuantificar duplicados

Detectar no es suficiente: antes de decidir qué hacer, conviene saber **cuántos** duplicados hay y **qué valores** se repiten.

### 4.1 Conteo con `.sum()`

`duplicated()` devuelve una `Series` booleana. Al aplicar `.sum()`, pandas trata `True` como 1 y `False` como 0, así que el resultado es el **número de filas marcadas**.

**Pregunta**: ¿`keep='first'` y `keep='last'` darán el mismo conteo? ¿Y `keep=False`?

In [ ]:
# Filas sobrantes (se conserva la primera aparición)
n_first = df.duplicated(keep='first').sum()

# Filas sobrantes (se conserva la última aparición)
n_last = df.duplicated(keep='last').sum()

# Todas las filas que pertenecen a algún grupo repetido
n_todas = df.duplicated(keep=False).sum()

print(f'keep="first": {n_first} filas sobrantes')
print(f'keep="last" : {n_last} filas sobrantes')
print(f'keep=False  : {n_todas} filas involucradas en duplicados')

**Observación**:

- `'first'` y `'last'` siempre dan el **mismo número** (9): cambian *cuáles* filas se marcan, no *cuántas*.
- `keep=False` da 14: son las 9 sobrantes más las 5 filas que se conservarían (una por cada grupo repetido: Ana, Juan, Luis, Carlos y Mario).
- Relación útil: `filas_involucradas − filas_sobrantes = número de grupos repetidos` → 14 − 9 = 5.

### 4.2 Proporción de duplicados con `.mean()`

La media de una `Series` booleana es la **proporción** de valores `True`. Sirve para expresar el problema en porcentaje, que es más comparable entre datasets de distinto tamaño.

In [ ]:
# Proporción de filas sobrantes respecto al total
proporcion = df.duplicated().mean()

print(f'Total de filas      : {len(df)}')
print(f'Filas duplicadas    : {df.duplicated().sum()}')
print(f'Proporción          : {proporcion:.4f}')
print(f'Porcentaje          : {proporcion * 100:.2f} %')

### 4.3 ¿Qué valores se repiten y cuántas veces? `value_counts()`

`value_counts()` (visto en sesiones anteriores) cuenta cuántas veces aparece cada valor. Aplicado a un `DataFrame`, cuenta cada **combinación de filas**; con el parámetro `subset` cuenta combinaciones de ciertas columnas.

In [ ]:
# Conteo de cada fila completa (combinación de todas las columnas)
print('Frecuencia de cada fila completa:')
print(df.value_counts())

In [ ]:
# Conteo por combinación de columnas específicas
conteo_nombre_edad = df.value_counts(subset=['Nombre', 'Edad'])

# Conservamos solo las combinaciones que aparecen más de una vez
repetidos = conteo_nombre_edad[conteo_nombre_edad > 1]

print('Combinaciones Nombre + Edad que se repiten:')
print(repetidos)
print('\nNúmero de grupos repetidos:', len(repetidos))

Con `normalize=True` se obtienen proporciones en lugar de conteos.

In [ ]:
# Proporción que representa cada ID dentro del total de filas
print(df['ID'].value_counts(normalize=True))

**Observación**: el `ID` 2 (Juan) representa el 25 % de las filas, aunque en realidad es solo uno de siete empleados (≈ 14.3 %). Así es como los duplicados sesgan las frecuencias.

## 5. Filtrar (ver) los duplicados

Antes de eliminar, **inspecciona** los registros. La `Series` booleana de `duplicated()` funciona como **máscara**: `df[mascara]` devuelve solo las filas donde la máscara es `True`.

| Expresión | Resultado |
|---|---|
| `df[df.duplicated(keep=False)]` | Todas las filas que forman parte de un grupo repetido |
| `df[df.duplicated()]` | Solo las filas sobrantes |
| `df[~df.duplicated(keep=False)]` | Solo las filas que **nunca** se repiten |
| `df[~df.duplicated()]` | Una fila por grupo (equivale a eliminar duplicados) |

El operador `~` **invierte** la máscara: `True` pasa a `False` y viceversa.

In [ ]:
# Guardamos la máscara en una variable para que el código sea legible
mascara_todos = df.duplicated(keep=False)

# Todas las filas involucradas en duplicados
print('Filas que pertenecen a un grupo repetido:')
print(df[mascara_todos])

Las filas repetidas aparecen dispersas: Ana está en las filas 0, 4 y 14. Para revisarlas conviene **ordenarlas**, de modo que las copias queden juntas.

### Método: `DataFrame.sort_values()`

```python
df.sort_values(by='columna', ascending=True)
```

Devuelve un **nuevo** `DataFrame` con las filas ordenadas según los valores de una o varias columnas. El original no cambia.

| Parámetro | Valores | Significado |
|---|---|---|
| `by` | nombre de columna o lista de columnas | Columna(s) que determinan el orden. Con una lista, ordena por la primera y, en caso de empate, por la segunda, etc. |
| `ascending` | `True` (por defecto) | Orden ascendente: de menor a mayor (A → Z, fechas antiguas → recientes). |
| | `False` | Orden descendente: de mayor a menor. |

Cada fila conserva su **índice original**, lo que permite saber de dónde venía.

**Pregunta**: al ordenar `df` por `Salario` en forma descendente, ¿quién aparecerá primero?

In [ ]:
# Ordenar por una columna, de mayor a menor
print(df.sort_values(by='Salario', ascending=False).head())

In [ ]:
# Ordenar por dos columnas: primero Edad; si hay empate, por Nombre
print(df.sort_values(by=['Edad', 'Nombre']).head(8))

**Observación**: Sara (70,000) aparece primero en el orden descendente, con su índice original 3. En el segundo caso las filas se agrupan por edad; los empates se resuelven por nombre.

Ahora aplicamos `sort_values()` a las filas duplicadas para ver cada grupo junto:

In [ ]:
# Mismas filas, ordenadas por ID para ver cada grupo junto
print(df[mascara_todos].sort_values(by='ID'))

In [ ]:
# ~ invierte la máscara: filas que aparecen una sola vez
print('Filas que nunca se repiten:')
print(df[~mascara_todos])

**Pregunta**: ¿qué devuelve `df[~df.duplicated()]` (sin `keep=False`)? ¿Cuántas filas tendrá?

In [ ]:
# ~ sobre keep='first': conserva la primera aparición de cada grupo
print(df[~df.duplicated()])
print('\nFilas:', len(df[~df.duplicated()]))

**Observación**: obtenemos 7 filas, una por empleado. Es exactamente la tabla limpia que construimos a mano en la sección 1.4. En la siguiente sección veremos el método dedicado para hacerlo: `drop_duplicates()`.

## 6. Eliminación con `drop_duplicates()`

### Método: `DataFrame.drop_duplicates()`

```python
df.drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
```

| Parámetro | Valores | Significado |
|---|---|---|
| `subset` | `None` o lista de columnas | Columnas que definen el duplicado (igual que en `duplicated()`). |
| `keep` | `'first'` (por defecto) | Conserva la primera aparición y elimina las demás. |
| | `'last'` | Conserva la última aparición. |
| | `False` | Elimina **todas** las filas de cada grupo repetido (no conserva ninguna). |
| `inplace` | `False` (por defecto) | Devuelve un **nuevo** `DataFrame`; el original no cambia. |
| | `True` | Modifica el `DataFrame` original y devuelve `None`. |
| `ignore_index` | `False` (por defecto) | Conserva los índices originales (quedan "huecos"). |
| | `True` | Renumera el índice como 0, 1, 2, … |

Relación con la sección anterior: `df.drop_duplicates(keep=k)` produce el mismo resultado que `df[~df.duplicated(keep=k)]`.

### 6.1 Uso básico

**Pregunta**: ¿cuántas filas tendrá el resultado? ¿Qué índices conservará?

In [ ]:
# Elimina duplicados exactos y conserva la primera aparición
df_limpio = df.drop_duplicates()

print(df_limpio)
print('\nFilas antes  :', len(df))
print('Filas después:', len(df_limpio))

# El DataFrame original NO cambia (inplace=False por defecto)
print('Filas en df (original):', len(df))

**Observación**: quedan 7 filas con índices `0, 1, 2, 3, 8, 9, 10`. Los índices originales se conservan; por eso hay saltos.

Comprobemos que la media del salario ahora coincide con la tabla construida a mano.

In [ ]:
print('Media con duplicados :', df['Salario'].mean())
print('Media sin duplicados :', df_limpio['Salario'].mean())

### 6.2 Comparar `keep='first'`, `keep='last'` y `keep=False`

In [ ]:
# Conserva la primera aparición de cada grupo
print("keep='first'")
print(df.drop_duplicates(keep='first'))

# Conserva la última aparición de cada grupo
print("\nkeep='last'")
print(df.drop_duplicates(keep='last'))

# Elimina TODAS las filas que tienen al menos una copia
print('\nkeep=False')
print(df.drop_duplicates(keep=False))

**Observación**:

- `'first'` y `'last'` devuelven los **mismos empleados** (7 filas), pero con **índices distintos**. En este caso no importa porque las copias son idénticas; importará cuando las copias difieran en columnas fuera de `subset` (sección 7.5).
- `keep=False` deja solo a Sara y Elena. **Pierde información**: Ana, Juan, Luis, Carlos y Mario desaparecen por completo. Úsalo únicamente cuando quieras aislar los registros que nunca se repiten, no para limpiar.

### 6.3 Eliminar con `subset` y renumerar con `ignore_index`

In [ ]:
# Un registro por ID; se conserva la primera aparición
# ignore_index=True renumera el índice 0, 1, 2, ...
personas_por_id = personas.drop_duplicates(subset=['ID'], ignore_index=True)
print(personas_por_id)

**Observación**: Vicente (ID 1) y Rogelio (ID 3) **desaparecieron** porque su `ID` ya había aparecido con Ana y Luis. `drop_duplicates()` no "sabe" que son personas distintas: aplica la regla que le damos con `subset`. Elegir mal las columnas elimina registros válidos.

### 6.4 El parámetro `inplace` y un error frecuente

Con `inplace=True` el método modifica el `DataFrame` original **y devuelve `None`**. Un error común es asignar ese resultado a una variable.

**Pregunta**: ¿qué contendrá la variable `resultado`?

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# Copia para no modificar df
df_copia = df.copy()

# ERROR INTENCIONAL: con inplace=True el método devuelve None
resultado = df_copia.drop_duplicates(inplace=True)

# Intentamos usar 'resultado' como si fuera un DataFrame
print(resultado.shape)

**Explicación del error**: `AttributeError: 'NoneType' object has no attribute 'shape'`.

Con `inplace=True`, `drop_duplicates()` limpia `df_copia` directamente y devuelve `None`; por eso `resultado` no es un `DataFrame`. Hay dos formas correctas (**usa solo una**):

```python
# Opción A (recomendada): asignar el resultado
df_copia = df_copia.drop_duplicates()

# Opción B: modificar en el lugar, sin asignar
df_copia.drop_duplicates(inplace=True)
```

Se recomienda la opción A: es más explícita y evita modificar datos sin querer.

In [ ]:
# Verificamos: df_copia sí fue modificado por la celda anterior
print(df_copia.shape)

### 6.5 Otro error frecuente: nombre de columna incorrecto en `subset`

Los nombres de columna distinguen mayúsculas y minúsculas.

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# ERROR INTENCIONAL: la columna se llama 'Nombre', no 'nombre'
df.drop_duplicates(subset=['nombre'])

**Explicación del error**: `KeyError: Index(['nombre'], dtype=...)`.

La columna `'nombre'` no existe. Antes de usar `subset`, consulta los nombres exactos con `df.columns`:

In [ ]:
# Lista de columnas disponibles
print(df.columns.tolist())

# Versión corregida
print(df.drop_duplicates(subset=['Nombre']))

## 7. Casos avanzados

En datos reales, `drop_duplicates()` rara vez se aplica "tal cual". Esta sección cubre situaciones frecuentes:

| # | Situación | Pregunta clave |
|---|---|---|
| 7.1 | Integración de fuentes (`pd.concat`) | ¿Cómo aparecen duplicados al unir tablas? |
| 7.2 | Duplicados ocultos por formato | ¿`"Ana"` y `"ana "` son lo mismo? |
| 7.3 | Valores faltantes (`NaN`) | ¿Dos `NaN` se consideran iguales? |
| 7.4 | Tipos de dato distintos | ¿`101` y `"101"` son lo mismo? |
| 7.5 | Conservar el registro más reciente | ¿Cuál copia se queda? |
| 7.6 | Conservar el registro más completo | ¿Cuál copia tiene más información? |
| 7.7 | Consolidar en lugar de eliminar (`groupby`) | ¿Se puede combinar la información de las copias? |
| 7.8 | Conflictos | Mismo `ID`, datos distintos: ¿eliminar o revisar? |

### 7.1 Duplicados por integración de fuentes

Una de las causas más comunes de duplicados es **unir tablas** que provienen de fuentes distintas: dos sucursales, dos meses de un mismo reporte, un archivo que se cargó dos veces. En pandas, las tablas con las mismas columnas se apilan con `pd.concat()`.

### 🔧 Función: `pd.concat()`

```python
pd.concat([df1, df2, ...], ignore_index=False)
```

Une varios `DataFrame` **uno debajo de otro** (apila sus filas) y devuelve un nuevo `DataFrame`.

| Parámetro | Valores | Significado |
|---|---|---|
| primer argumento | **lista** de `DataFrame` | Tablas que se unen, en ese orden. Deben ir entre corchetes. |
| `ignore_index` | `False` (por defecto) | Conserva el índice de cada tabla; puede haber índices repetidos. |
| | `True` | Renumera el índice del resultado como 0, 1, 2, … |

- Las columnas se alinean **por nombre**. Si una tabla tiene una columna que la otra no, en las filas de esta última aparece `NaN`.
- `pd.concat()` **no elimina** duplicados: si la misma fila está en ambas tablas, aparecerá dos veces.

Caso: una tienda tiene dos sucursales que exportan su catálogo de clientes. Algunos clientes compran en ambas.

In [ ]:
# Catálogo de la sucursal Norte
sucursal_norte = pd.DataFrame({
    'Cliente': ['C01', 'C02', 'C03'],
    'Nombre':  ['Ana', 'Juan', 'Luis'],
    'Ciudad':  ['Juárez', 'Juárez', 'Chihuahua']
})

# Catálogo de la sucursal Sur (C02 y C03 también están en la Norte)
sucursal_sur = pd.DataFrame({
    'Cliente': ['C02', 'C04', 'C03'],
    'Nombre':  ['Juan', 'Sara', 'Luis'],
    'Ciudad':  ['Juárez', 'Delicias', 'Chihuahua']
})

print(sucursal_norte)
print()
print(sucursal_sur)

**Pregunta**: al unir ambas tablas con `pd.concat()` sin más argumentos, ¿cuántas filas tendrá el resultado? ¿Qué valores tendrá el índice?

In [ ]:
# Unión SIN renumerar el índice
union = pd.concat([sucursal_norte, sucursal_sur])
print(union)

**Observación**: el resultado tiene 6 filas y el índice se repite (`0, 1, 2, 0, 1, 2`), porque cada tabla conserva su propio índice. Esto tiene dos consecuencias:

1. **Índice duplicado ≠ fila duplicada**. `duplicated()` compara **valores**, no el índice. Para detectar índices repetidos se usa `df.index.duplicated()`.
2. Un índice repetido hace ambigua la selección: `union.loc[0]` devuelve **dos** filas.

In [ ]:
# Índices repetidos (compara solo el índice)
print('Índices repetidos:', union.index.duplicated().sum())

# Filas repetidas (compara los valores de las columnas)
print('Filas repetidas  :', union.duplicated().sum())

# Selección ambigua: .loc[0] devuelve más de una fila
print('\nunion.loc[0]:')
print(union.loc[0])

Por eso, al unir tablas conviene usar `ignore_index=True` y, después, buscar los duplicados reales.

In [ ]:
# Unión renumerando el índice
union = pd.concat([sucursal_norte, sucursal_sur], ignore_index=True)
print(union)

# Duplicados reales: clientes que estaban en ambas sucursales
print('\nClientes en ambas sucursales:')
print(union[union.duplicated(keep=False)].sort_values(by='Cliente'))

# Catálogo único
catalogo = union.drop_duplicates(ignore_index=True)
print('\nCatálogo único:')
print(catalogo)

**Observación**: C02 y C03 aparecían en ambas sucursales; tras `drop_duplicates()` el catálogo queda con 4 clientes. **Buena práctica**: después de cualquier `pd.concat()`, revisa duplicados.

### 7.2 Duplicados ocultos: diferencias de formato

`duplicated()` compara valores **exactamente**, carácter por carácter. `"Ana López"`, `"ana lópez"` y `"Ana López "` (con espacio final) son, para pandas, tres textos distintos.

**Pregunta**: ¿cuántos duplicados exactos detectará `duplicated()` en la siguiente tabla?

In [ ]:
# Registro de clientes capturado por distintas personas
clientes = pd.DataFrame({
    'Nombre': ['Ana López', 'ana lópez', 'Ana López ', 'Juan Pérez', 'JUAN PÉREZ', 'Luis Díaz'],
    'Correo': ['ana@mail.com', 'ANA@mail.com', 'ana@mail.com', ' juan@mail.com', 'juan@mail.com', 'luis@mail.com'],
    'Ciudad': ['Juárez', 'juárez', 'Juárez', 'Chihuahua', 'Chihuahua', 'Juárez']
})

print(clientes)
print('\nDuplicados exactos detectados:', clientes.duplicated().sum())

**Observación**: pandas no detecta ninguno, aunque a simple vista hay solo tres clientes.

**Solución**: **normalizar** el texto antes de comparar.

### Accesor `.str`: métodos de texto sobre una columna

En Python, una cadena tiene métodos como `'  Ana '.strip()` o `'ANA'.lower()`. Una columna de pandas (`Series`) **no** es una cadena, sino un conjunto de cadenas. El **accesor** `.str` permite aplicar un método de texto a **cada elemento** de la columna a la vez, sin escribir un ciclo:

```python
df['columna'].str.metodo()
```

El resultado es una nueva `Series` del mismo tamaño. Los dos métodos que usaremos:

| Método | Efecto | Ejemplo |
|---|---|---|
| `.str.strip()` | Quita espacios al inicio y al final | `' juan@mail.com'` → `'juan@mail.com'` |
| `.str.lower()` | Convierte a minúsculas | `'JUAN PÉREZ'` → `'juan pérez'` |

Pueden **encadenarse**: `serie.str.strip().str.lower()` primero quita espacios y, sobre ese resultado, convierte a minúsculas.

 **Pregunta**: ¿qué imprimirá `repr()` para cada valor antes y después de normalizar? (`repr()` muestra las comillas, lo que permite ver los espacios.)

In [ ]:
# Columna de ejemplo con espacios y mayúsculas
nombres = pd.Series(['Ana López', 'ana lópez', 'Ana López ', ' JUAN PÉREZ'])

# .str aplica el método a cada elemento de la Series
normalizados = nombres.str.strip().str.lower()

# repr() muestra las comillas para hacer visibles los espacios
for original, limpio in zip(nombres, normalizados):
    print(repr(original), '→', repr(limpio))

**Observación**: los cuatro valores terminan como dos textos distintos (`'ana lópez'` y `'juan pérez'`). Aplicamos lo mismo a las columnas de `clientes`:

In [ ]:
# Copia para normalizar sin alterar los datos originales
clientes_norm = clientes.copy()

# Normalizamos cada columna de texto: sin espacios extremos y en minúsculas
for columna in ['Nombre', 'Correo', 'Ciudad']:
    clientes_norm[columna] = clientes_norm[columna].str.strip().str.lower()

print(clientes_norm)
print('\nDuplicados detectados tras normalizar:', clientes_norm.duplicated().sum())

Ahora hay dos opciones para eliminar:

- **Opción A**: quedarse con la versión normalizada (`clientes_norm.drop_duplicates()`).
- **Opción B**: usar la máscara calculada sobre los datos normalizados para filtrar los datos **originales**, y así conservar el formato tal como se capturó la primera vez.

> `reset_index(drop=True)` renumera el índice como 0, 1, 2, … y descarta el índice anterior (`drop=True` evita que se agregue como columna). Tiene el mismo efecto que `ignore_index=True` en `drop_duplicates()`, pero se puede aplicar a cualquier `DataFrame`, por ejemplo, al resultado de un filtrado.

In [ ]:
# Opción A: resultado con texto normalizado
opcion_a = clientes_norm.drop_duplicates(ignore_index=True)
print('Opción A (normalizado):')
print(opcion_a)

# Opción B: la máscara se calcula sobre la versión normalizada,
# pero se aplica al DataFrame original (mismo índice en ambos)
mascara_norm = clientes_norm.duplicated()
opcion_b = clientes[~mascara_norm].reset_index(drop=True)
print('\nOpción B (formato original de la primera aparición):')
print(opcion_b)

**Observación**: ambas opciones dejan 3 clientes. La opción B funciona porque `clientes` y `clientes_norm` comparten el mismo índice, de modo que la máscara de uno se puede aplicar al otro.

**Limitación**: `strip()` y `lower()` no resuelven acentos (`"Juárez"` vs `"Juarez"`), abreviaturas (`"Cd. Juárez"` vs `"Ciudad Juárez"`) ni errores tipográficos. Esos casos requieren reglas de reemplazo o técnicas de coincidencia aproximada, que están fuera del alcance de esta sesión.

### 7.3 Valores faltantes (`NaN`)

**Pregunta**: dos filas que tienen `NaN` en la misma columna y los demás valores iguales, ¿se consideran duplicadas?

In [ ]:
import numpy as np

# np.nan representa un valor faltante
contactos = pd.DataFrame({
    'Nombre':   ['Ana', 'Ana', 'Juan', 'Juan'],
    'Telefono': ['656-111-2233', '656-111-2233', np.nan, np.nan]
})

print(contactos)
print('\nduplicated():')
print(contactos.duplicated())

**Observación**: sí. `duplicated()` y `drop_duplicates()` tratan todos los `NaN` como **iguales entre sí**. Esto es distinto de la comparación con `==`, donde `np.nan == np.nan` da `False`.

**Implicación**: si varias filas solo tienen `NaN` en las columnas de `subset`, pandas las tratará como un único registro y eliminará las demás. Por ejemplo, detectar duplicados por `Telefono` eliminaría a todos los clientes sin teléfono excepto uno.

In [ ]:
# Ana se registró dos veces con el mismo teléfono;
# Sara, Luis y Elena son personas DISTINTAS sin teléfono registrado
agenda = pd.DataFrame({
    'Nombre':   ['Ana', 'Ana', 'Sara', 'Luis', 'Elena'],
    'Telefono': ['656-111-2233', '656-111-2233', np.nan, np.nan, np.nan]
})

# Eliminar duplicados solo por Telefono: se pierden Luis y Elena
print(agenda.drop_duplicates(subset=['Telefono']))

**Observación**: la copia de Ana se eliminó correctamente, pero también desaparecieron Luis y Elena: su `NaN` se consideró igual al de Sara.

**Buena práctica**: marcar como duplicado solo si la fila se repite **y** tiene valor en la columna clave. Se combinan dos máscaras con `&` (y lógico):

- `agenda.duplicated(subset=['Telefono'])`: `True` si el teléfono ya apareció (incluye los `NaN` repetidos).
- `agenda['Telefono'].notna()`: `True` si **hay** teléfono. (`.isna()` hace lo contrario: `True` donde hay `NaN`.)

In [ ]:
# Duplicado real = teléfono repetido Y teléfono presente
duplicado_real = agenda.duplicated(subset=['Telefono']) & agenda['Telefono'].notna()

# Vemos las máscaras lado a lado
revision_nan = agenda.copy()
revision_nan['tel_repetido']   = agenda.duplicated(subset=['Telefono'])
revision_nan['tiene_tel']      = agenda['Telefono'].notna()
revision_nan['duplicado_real'] = duplicado_real
print(revision_nan)

# Conservamos todo lo que NO es un duplicado real
agenda_limpia = agenda[~duplicado_real].reset_index(drop=True)
print('\nResultado:')
print(agenda_limpia)

**Observación**: solo se eliminó la segunda fila de Ana. Sara, Luis y Elena se conservan.

**Alternativa con `pd.concat()`**: separar las filas con y sin teléfono, deduplicar solo las primeras y volver a unir ambas partes.

In [ ]:
# Parte 1: filas con teléfono, deduplicadas por Telefono
con_tel = agenda[agenda['Telefono'].notna()].drop_duplicates(subset=['Telefono'])

# Parte 2: filas sin teléfono, se conservan todas
sin_tel = agenda[agenda['Telefono'].isna()]

# Unimos ambas partes en un solo DataFrame
agenda_limpia_2 = pd.concat([con_tel, sin_tel], ignore_index=True)
print(agenda_limpia_2)

**Observación**: el resultado es el mismo. La versión con máscaras conserva el orden original de las filas; la versión con `pd.concat()` coloca primero las filas con teléfono. Ambas son válidas.

### 7.4 Tipos de dato distintos

Cuando una columna mezcla tipos (por ejemplo, números y textos, algo frecuente al combinar archivos), valores que "se ven iguales" pueden no serlo.

**Pregunta**: ¿cuántas filas marcará `duplicated()`?

In [ ]:
# La columna Codigo mezcla enteros y cadenas de texto
productos = pd.DataFrame({
    'Codigo':   [101, '101', 102, '102'],
    'Producto': ['Teclado', 'Teclado', 'Mouse', 'Mouse']
})

print(productos)
print('\nTipo de cada valor en Codigo:')
print(productos['Codigo'].apply(type))
print('\nDuplicados detectados:', productos.duplicated().sum())

**Observación**: ninguno. El entero `101` y el texto `'101'` son valores distintos. Por eso el **ajuste de tipos** (visto en la etapa de profiling) debe hacerse **antes** de buscar duplicados.

In [ ]:
productos_tipos = productos.copy()

# Unificamos el tipo: toda la columna como entero
productos_tipos['Codigo'] = productos_tipos['Codigo'].astype(int)

print('Duplicados tras ajustar el tipo:', productos_tipos.duplicated().sum())
print(productos_tipos.drop_duplicates())

### 7.5 Conservar el registro más reciente

Cuando las copias difieren en columnas **fuera** de `subset`, la elección de `keep` decide **qué información sobrevive**. `keep='first'` y `keep='last'` dependen del **orden de las filas**, no de la fecha. Si el `DataFrame` no está ordenado, "el último" puede no ser el más reciente.

Caso: un sistema guarda un registro nuevo cada vez que un cliente actualiza su dirección. Queremos **la dirección vigente** de cada cliente.

In [ ]:
# Historial de direcciones; las filas NO están ordenadas por fecha
direcciones = pd.DataFrame({
    'Cliente':   ['C01', 'C02', 'C01', 'C03', 'C02', 'C01'],
    'Direccion': ['Av. Juárez 10', 'Calle 5 #20', 'Blvd. Tomás Fernández 300',
                  'Av. Tecnológico 1340', 'Calle 8 #45', 'Av. de la Raza 77'],
    'Fecha':     ['2025-01-15', '2024-06-01', '2026-03-10',
                  '2025-11-20', '2026-01-05', '2025-08-30']
})

print(direcciones)

# Tipo de dato de cada columna: Fecha es TEXTO, no fecha
print('\nTipos de dato:')
print(direcciones.dtypes)

La columna `Fecha` contiene **texto**. Para ordenar cronológicamente, primero hay que convertirla al tipo fecha.

### Función: `pd.to_datetime()`

```python
pd.to_datetime(serie, format=None)
```

Convierte una columna de texto (o números) en valores de tipo **fecha** (`datetime64`). Con fechas, pandas puede ordenar cronológicamente, comparar (`>`, `<`) y calcular diferencias de tiempo.

| Parámetro | Valores | Significado |
|---|---|---|
| `serie` | `Series` con fechas en texto | Columna que se convierte. |
| `format` | `None` (por defecto) | pandas infiere el formato (funciona bien con `'AAAA-MM-DD'`). |
| | cadena de formato, p. ej. `'%d/%m/%Y'` | Indica el formato exacto: `%d` día, `%m` mes, `%Y` año de 4 dígitos. |

¿Por qué no ordenar el texto directamente? El texto se ordena **carácter por carácter**, como en un diccionario. Con el formato `AAAA-MM-DD` eso coincide con el orden cronológico, pero con otros formatos no:

In [ ]:
# Fechas en formato día/mes/año guardadas como texto
fechas_texto = pd.Series(['15/01/2025', '02/12/2026', '28/03/2024'])

# Orden como TEXTO: compara '0' < '1' < '2' (el día), no la fecha
print('Ordenado como texto:')
print(fechas_texto.sort_values())

# Orden como FECHA: se indica el formato día/mes/año
fechas = pd.to_datetime(fechas_texto, format='%d/%m/%Y')
print('\nOrdenado como fecha:')
print(fechas.sort_values())

**Observación**: como texto, el orden resulta `02/12/2026`, `15/01/2025`, `28/03/2024`: se ordenó por el **día** (`02 < 15 < 28`) y quedó exactamente al revés del orden cronológico. Como fecha, el orden es correcto: 2024, 2025, 2026. Además, `pd.to_datetime` muestra las fechas en el formato estándar `AAAA-MM-DD`.

Convertimos la columna `Fecha` de `direcciones`:

In [ ]:
# Convertimos Fecha a tipo fecha para que el orden sea cronológico
direcciones['Fecha'] = pd.to_datetime(direcciones['Fecha'])

print(direcciones.dtypes)

 **Pregunta**: si aplicamos `drop_duplicates(subset=['Cliente'], keep='last')` **sin ordenar**, ¿qué dirección quedará para `C01`? ¿Es la más reciente?

In [ ]:
# INCORRECTO: keep='last' toma la última FILA, no la fecha más reciente
sin_ordenar = direcciones.drop_duplicates(subset=['Cliente'], keep='last')
print(sin_ordenar)

**Observación**: para `C01` quedó `Av. de la Raza 77` (2025-08-30), pero su dirección más reciente es `Blvd. Tomás Fernández 300` (2026-03-10). El resultado es incorrecto porque la última fila no es la más reciente.

**Solución**: ordenar por fecha con `sort_values()` **antes** de eliminar.



---


### Encadenamiento de métodos


---



Como `sort_values()`, `drop_duplicates()` y `reset_index()` devuelven un nuevo `DataFrame`, se pueden escribir uno tras otro: cada método trabaja sobre el resultado del anterior. Estas dos versiones son equivalentes:

```python
# Paso a paso, con variables intermedias
paso1 = direcciones.sort_values(by='Fecha')
paso2 = paso1.drop_duplicates(subset=['Cliente'], keep='last')

# Encadenado
resultado = direcciones.sort_values(by='Fecha').drop_duplicates(subset=['Cliente'], keep='last')
```

Cuando la cadena es larga, se encierra entre **paréntesis** para escribir un método por línea y comentar cada paso.

In [ ]:
# CORRECTO: 1) ordenar cronológicamente, 2) conservar la última aparición de cada cliente
vigentes = (
    direcciones
    .sort_values(by='Fecha')                          # de la más antigua a la más reciente
    .drop_duplicates(subset=['Cliente'], keep='last')  # la última ahora ES la más reciente
    .sort_values(by='Cliente')                        # opcional: ordenar para presentar
    .reset_index(drop=True)
)

print(vigentes)

### 7.6 Conservar el registro más completo

Otra regla frecuente: entre las copias de un mismo registro, conservar la que tiene **menos valores faltantes**.

Estrategia:

1. Calcular cuántos valores **no nulos** tiene cada fila: `df.notna().sum(axis=1)`.
   - `notna()` devuelve `True` donde hay dato.
   - `sum(axis=1)` suma **por fila** (a lo ancho), no por columna.
2. Ordenar de mayor a menor completitud.
3. Eliminar duplicados conservando la **primera** aparición (la más completa).
4. Quitar la columna auxiliar.

In [ ]:
# Mismo alumno capturado en distintos momentos, con datos incompletos
alumnos = pd.DataFrame({
    'Matricula': ['A100', 'A100', 'A200', 'A200', 'A300'],
    'Nombre':    ['Sofía', 'Sofía', 'Diego', 'Diego', 'Elena'],
    'Correo':    [np.nan, 'sofia@uacj.mx', 'diego@uacj.mx', 'diego@uacj.mx', 'elena@uacj.mx'],
    'Telefono':  [np.nan, '656-222-3344', np.nan, '656-555-6677', np.nan]
})

# Paso 1: número de valores no nulos por fila
alumnos['completitud'] = alumnos.notna().sum(axis=1)
print(alumnos)

In [ ]:
mas_completos = (
    alumnos
    .sort_values(by='completitud', ascending=False)        # Paso 2: más completos primero
    .drop_duplicates(subset=['Matricula'], keep='first')   # Paso 3: conservar el más completo
    .drop(columns=['completitud'])                         # Paso 4: quitar columna auxiliar
    .sort_values(by='Matricula')
    .reset_index(drop=True)
)

print(mas_completos)

**Observación**: para `A100` se conservó la fila con correo y teléfono; para `A200`, la que tiene teléfono. Con `keep='first'` sin ordenar habríamos conservado las filas con más faltantes.

### 7.7 Consolidar en lugar de eliminar: `groupby()`

`drop_duplicates()` siempre conserva **una fila completa** y descarta las demás. Pero a veces cada copia tiene **una parte** de la información: una tiene el correo y otra el teléfono. Si conservamos solo una, perdemos datos.

**Pregunta**: en la siguiente tabla, ¿qué información se pierde si aplicamos la estrategia de 7.6 (conservar la fila más completa)?

In [ ]:
# Cada copia tiene información que la otra no tiene
contactos_alumnos = pd.DataFrame({
    'Matricula': ['A100', 'A200', 'A100', 'A300', 'A200'],
    'Nombre':    ['Sofía', 'Diego', 'Sofía', 'Elena', 'Diego'],
    'Correo':    ['sofia@uacj.mx', np.nan, np.nan, 'elena@uacj.mx', 'diego@uacj.mx'],
    'Telefono':  [np.nan, '656-555-6677', '656-222-3344', np.nan, np.nan]
})

print(contactos_alumnos)

**Observación**: las dos filas de `A100` tienen la misma completitud (3 valores). Conservar cualquiera de ellas pierde el correo **o** el teléfono de Sofía. Lo mismo ocurre con Diego. La solución es **consolidar**: construir un registro por matrícula que combine los datos de todas sus copias.

### Método: `DataFrame.groupby()`

```python
df.groupby('columna_clave').funcion_de_agregacion()
```

`groupby()` sigue el esquema **dividir → aplicar → combinar**:

1. **Dividir**: separa las filas en grupos según los valores de la columna clave (todas las filas de `A100` en un grupo, las de `A200` en otro…).
2. **Aplicar**: calcula una función sobre cada grupo.
3. **Combinar**: une los resultados en una tabla con **una fila por grupo**.

Por sí solo, `df.groupby('col')` no muestra nada: hay que indicar qué calcular. Funciones útiles para duplicados:

| Función | Resultado por grupo |
|---|---|
| `.size()` | Número de filas del grupo (cuántas copias hay) |
| `.first()` | Primer valor **no nulo** de cada columna |
| `.last()` | Último valor **no nulo** de cada columna |
| `.max()` / `.min()` | Valor máximo / mínimo de cada columna (útil con fechas) |
| `.nunique()` | Número de valores distintos de cada columna |

- La columna clave pasa a ser el **índice** del resultado. Con `.reset_index()` vuelve a ser una columna normal.
- Para calcular solo sobre una columna: `df.groupby('clave')['columna'].funcion()`.

In [ ]:
# Paso 1: ¿cuántas copias tiene cada matrícula?
print('Filas por matrícula:')
print(contactos_alumnos.groupby('Matricula').size())

In [ ]:
# Paso 2: consolidar; .first() toma el primer valor NO nulo de cada columna en cada grupo
consolidado = contactos_alumnos.groupby('Matricula').first()
print('Resultado de groupby (Matricula es el índice):')
print(consolidado)

# Paso 3: reset_index() convierte el índice Matricula en columna
consolidado = consolidado.reset_index()
print('\nCon reset_index():')
print(consolidado)

**Observación**: Sofía y Diego ahora tienen correo **y** teléfono, aunque ninguna fila original los tenía juntos. `.first()` ignora los `NaN` y toma, columna por columna, el primer valor disponible.

**Precaución**: si las copias tienen valores **distintos** (dos correos diferentes), `.first()` elige uno sin avisar. Antes de consolidar, verifica que no haya conflictos (sección 7.8).

`groupby()` también sirve para comprobar el resultado de 7.5: la fecha más reciente de cada cliente debe coincidir con la de `vigentes`.

In [ ]:
# Fecha máxima (más reciente) por cliente, calculada solo sobre la columna Fecha
print(direcciones.groupby('Cliente')['Fecha'].max())

### 7.8 Conflictos: mismo identificador, datos distintos

Hasta ahora hemos supuesto que, si el identificador coincide, las copias representan la misma entidad. No siempre es así. Si dos filas comparten `ID` pero tienen **datos contradictorios**, eliminar una de ellas oculta un problema de calidad.

Se puede distinguir entre:

- **Duplicado exacto**: mismo `ID` y mismos datos → se puede eliminar con seguridad.
- **Conflicto**: mismo `ID`, datos distintos → requiere revisión.

Combinamos dos máscaras con el operador `&` (y lógico):

In [ ]:
empleados = pd.DataFrame({
    'ID':      [1, 2, 3, 1, 3, 4, 5, 5],
    'Nombre':  ['Ana', 'Juan', 'Luis', 'Ana', 'Luis', 'Sara', 'Carlos', 'Karla'],
    'Edad':    [25, 30, 35, 25, 36, 40, 28, 31],
    'Salario': [50000, 60000, 55000, 50000, 55000, 70000, 48000, 52000]
})

# Filas cuyo ID aparece más de una vez
id_repetido = empleados.duplicated(subset=['ID'], keep=False)

# Filas que son copia exacta de otra
fila_repetida = empleados.duplicated(keep=False)

# Conflicto: el ID se repite, pero la fila completa NO se repite
conflicto = id_repetido & ~fila_repetida

# Mostramos todas las máscaras juntas para interpretarlas
revision = empleados.copy()
revision['id_repetido']   = id_repetido
revision['fila_repetida'] = fila_repetida
revision['conflicto']     = conflicto
print(revision)

**Observación**:

- `ID 1` (Ana): copia exacta → `conflicto = False`. Se puede eliminar una.
- `ID 3` (Luis): la edad difiere (35 vs 36) → conflicto. ¿Error de captura o actualización?
- `ID 5`: `Carlos` y `Karla` → conflicto grave. Probablemente son dos personas con un `ID` mal asignado.

Con `groupby()` y `.nunique()` podemos ver **en qué columnas** difiere cada identificador. Un valor mayor que 1 indica que ese `ID` tiene más de un valor distinto en esa columna.

In [ ]:
# Número de valores distintos por columna, para cada ID
diagnostico = empleados.groupby('ID').nunique()
print(diagnostico)

# IDs con al menos una columna con más de un valor distinto
# (diagnostico > 1) produce True/False; .any(axis=1) revisa cada fila
ids_en_conflicto = diagnostico[(diagnostico > 1).any(axis=1)]
print('\nIDs con conflicto y columnas afectadas:')
print(ids_en_conflicto)

**Observación**: el `ID 3` solo difiere en `Edad`; el `ID 5` difiere en `Nombre`, `Edad` y `Salario`. Este diagnóstico ayuda a decidir: una edad distinta puede corregirse; un nombre distinto sugiere un `ID` mal asignado.

Un flujo prudente separa ambos casos: elimina los duplicados exactos y envía los conflictos a revisión.

In [ ]:
# 1) Eliminar solo duplicados exactos
empleados_limpio = empleados.drop_duplicates(ignore_index=True)

# 2) Aislar los conflictos para revisión manual (se calculan de nuevo sobre el resultado)
en_conflicto = empleados_limpio[empleados_limpio.duplicated(subset=['ID'], keep=False)]

print('Tras eliminar duplicados exactos:', len(empleados_limpio), 'filas')
print('\nRegistros en conflicto para revisar:')
print(en_conflicto.sort_values(by='ID'))

## 8. Caso real: Customer Personality Analysis

Aplicaremos lo anterior al dataset **Customer Personality Analysis** (2,240 clientes de una empresa de venta al por menor), ya utilizado en sesiones anteriores. El archivo usa `;` como separador.

**Pregunta**: la columna `ID` identifica a cada cliente. ¿Cuántos duplicados exactos esperas encontrar?

In [ ]:
# Carga del dataset desde GitHub (separador: punto y coma)
url = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'
clientes_mkt = pd.read_csv(url, sep=';')

print('Dimensiones:', clientes_mkt.shape)
print('Duplicados exactos:', clientes_mkt.duplicated().sum())
print('IDs repetidos    :', clientes_mkt.duplicated(subset=['ID']).sum())

**Observación**: cero duplicados exactos y cero `ID` repetidos. ¿Significa que el dataset está limpio?

No necesariamente. Si el mismo cliente se registró dos veces, el sistema pudo asignarle **dos `ID` distintos**. Como el `ID` es diferente, la fila completa nunca coincide. Hay que comparar **todas las columnas excepto `ID`**.

`df.columns.drop('ID')` devuelve la lista de columnas sin `ID`, lista para usarse como `subset`.

In [ ]:
# Todas las columnas excepto el identificador
columnas_sin_id = clientes_mkt.columns.drop('ID')
print('Columnas comparadas:', len(columnas_sin_id))

# Duplicados considerando todo menos el ID
n_sobrantes    = clientes_mkt.duplicated(subset=columnas_sin_id).sum()
n_involucradas = clientes_mkt.duplicated(subset=columnas_sin_id, keep=False).sum()

print(f'Filas sobrantes          : {n_sobrantes}')
print(f'Filas involucradas       : {n_involucradas}')
print(f'Grupos repetidos         : {n_involucradas - n_sobrantes}')
print(f'Porcentaje de sobrantes  : {n_sobrantes / len(clientes_mkt) * 100:.2f} %')

**Observación**: 182 filas (≈ 8.1 %) son copias de otro cliente con un `ID` distinto. Coinciden en las 28 columnas restantes: año de nacimiento, educación, estado civil, ingreso, fecha de alta, gastos por categoría y respuestas a campañas. Es muy poco probable que se trate de clientes diferentes.

Inspeccionemos algunos grupos, ordenados para que las copias queden juntas:

In [ ]:
# Filas involucradas, ordenadas para ver cada grupo junto
mascara_mkt = clientes_mkt.duplicated(subset=columnas_sin_id, keep=False)

columnas_vista = ['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Dt_Customer', 'MntWines']

print(
    clientes_mkt[mascara_mkt]
    .sort_values(by=['Year_Birth', 'Income', 'Dt_Customer'])
    [columnas_vista]
    .head(10)
)

¿Cuántas veces se repite cada grupo? Contamos el tamaño de cada combinación con `value_counts()` y luego contamos esos tamaños.

In [ ]:
# Tamaño de cada combinación de columnas (sin ID)
tamanos = clientes_mkt.value_counts(subset=list(columnas_sin_id), dropna=False)

# Nos quedamos con los grupos repetidos y contamos cuántos hay de cada tamaño
print('Número de grupos según cuántas veces aparecen:')
print(tamanos[tamanos > 1].value_counts().sort_index())

> `dropna=False` hace que `value_counts()` también cuente las combinaciones que contienen `NaN` (la columna `Income` tiene faltantes). Sin este argumento se omitirían.

Ahora medimos el efecto de los duplicados en algunas métricas descriptivas y eliminamos.

In [ ]:
# Eliminamos las copias, conservando la primera aparición de cada cliente
clientes_mkt_limpio = clientes_mkt.drop_duplicates(subset=columnas_sin_id, keep='first', ignore_index=True)

print('Filas antes  :', len(clientes_mkt))
print('Filas después:', len(clientes_mkt_limpio))

# Comparación de métricas antes y después
print(f"\nIngreso medio      antes: {clientes_mkt['Income'].mean():,.2f}")
print(f"Ingreso medio    después: {clientes_mkt_limpio['Income'].mean():,.2f}")
print(f"\nTasa de respuesta  antes: {clientes_mkt['Response'].mean():.4f}")
print(f"Tasa de respuesta después: {clientes_mkt_limpio['Response'].mean():.4f}")

# Verificación: ya no deben quedar duplicados con ese criterio
print('\nDuplicados restantes:', clientes_mkt_limpio.duplicated(subset=columnas_sin_id).sum())

**Observación**: las diferencias en las medias son pequeñas porque los duplicados representan ~8 % de los datos y están distribuidos en todo el rango. Aun así:

- Cualquier **conteo** (número de clientes, clientes por segmento) estaba inflado en 182 unidades.
- Si este dataset se usara para entrenar un modelo que prediga `Response`, un cliente podría quedar en entrenamiento y su copia en prueba, lo que inflaría la exactitud reportada.

**Decisión documentada**: en un proyecto real, se debe registrar el criterio usado (*"se consideran duplicados los registros que coinciden en todas las columnas excepto `ID`; se conserva la primera aparición"*) para que el análisis sea reproducible.

## 9. Cierre

### Resumen de métodos

| Tarea | Código |
|---|---|
| Detectar duplicados exactos | `df.duplicated()` |
| Detectar por columnas clave | `df.duplicated(subset=['c1', 'c2'])` |
| Marcar todas las ocurrencias | `df.duplicated(keep=False)` |
| Contar filas sobrantes | `df.duplicated().sum()` |
| Proporción de sobrantes | `df.duplicated().mean()` |
| Frecuencia de combinaciones | `df.value_counts(subset=['c1', 'c2'])` |
| Ver los grupos repetidos | `df[df.duplicated(keep=False)].sort_values(by='c1')` |
| Ver filas que nunca se repiten | `df[~df.duplicated(keep=False)]` |
| Eliminar conservando la primera | `df.drop_duplicates()` |
| Eliminar conservando la última | `df.drop_duplicates(keep='last')` |
| Eliminar y renumerar | `df.drop_duplicates(ignore_index=True)` |
| Conservar el más reciente | `df.sort_values('fecha').drop_duplicates(subset=['id'], keep='last')` |
| Conservar el más completo | ordenar por `df.notna().sum(axis=1)` y usar `keep='first'` |
| Excluir `NaN` de la clave | `df[~(df.duplicated(subset=['c1']) & df['c1'].notna())]` |
| Normalizar texto | `df['c1'].str.strip().str.lower()` |
| Convertir a fecha | `pd.to_datetime(df['fecha'])` |
| Ordenar | `df.sort_values(by='c1', ascending=True)` |
| Unir tablas | `pd.concat([df1, df2], ignore_index=True)` |
| Detectar índices repetidos | `df.index.duplicated()` |
| Copias por clave | `df.groupby('id').size()` |
| Consolidar copias | `df.groupby('id').first().reset_index()` |
| Columnas en conflicto por clave | `df.groupby('id').nunique()` |
| Detectar conflictos | `df.duplicated(subset=['id'], keep=False) & ~df.duplicated(keep=False)` |

### Flujo recomendado

1. Si los datos vienen de varias fuentes, **unirlos** con `pd.concat(..., ignore_index=True)`.
2. **Ajustar tipos** y **normalizar texto** (espacios, mayúsculas).
3. **Definir el criterio** de duplicado (`subset`): ¿qué columnas identifican un registro?
4. **Cuantificar** (`.sum()`, `.mean()`, `value_counts()`).
5. **Inspeccionar** los grupos (`keep=False` + `sort_values()`).
6. **Separar** duplicados exactos de conflictos.
7. **Decidir qué copia conservar** (primera, última, más reciente, más completa) y ordenar en consecuencia, o **consolidar** con `groupby()`.
8. **Eliminar** con `drop_duplicates()` asignando el resultado a una nueva variable.
9. **Verificar** que `duplicated().sum()` sea 0 y **documentar** el criterio.

### Buenas prácticas

**Lo que se puede hacer**

- Guardar el resultado en una variable nueva: `df_limpio = df.drop_duplicates()`.
- Usar `subset` con columnas que identifican el registro: `subset=['Matricula']`, `subset=['Correo', 'Fecha']`.
- Excluir identificadores generados por el sistema cuando se sospecha doble registro: `subset=df.columns.drop('ID')`.
- Ordenar antes de usar `keep`: `df.sort_values('Fecha').drop_duplicates(subset=['Cliente'], keep='last')`.
- Revisar duplicados después de unir tablas: `pd.concat([df1, df2], ignore_index=True).duplicated().sum()`.
- Consolidar cuando cada copia aporta datos distintos: `df.groupby('Matricula').first()`.
- Comparar métricas antes y después de eliminar: `df['Salario'].mean()` vs `df_limpio['Salario'].mean()`.

**Lo que no se puede hacer**

- Asignar el resultado de `inplace=True`: `df2 = df.drop_duplicates(inplace=True)` deja `df2 = None`.
- Usar `keep=False` para "limpiar": `df.drop_duplicates(keep=False)` elimina también la copia que debía conservarse.
- Deduplicar por una sola columna poco informativa: `subset=['Edad']` elimina personas distintas con la misma edad.
- Unir tablas sin `ignore_index=True` y luego seleccionar con `.loc`: `pd.concat([df1, df2]).loc[0]` devuelve varias filas.
- Consolidar con `.first()` sin revisar conflictos: si hay dos correos distintos, elige uno sin avisar.
- Confiar en `keep='last'` como "más reciente" sin ordenar por fecha.
- Deduplicar por una columna con `NaN` sin excluir los faltantes: `df.drop_duplicates(subset=['Telefono'])` trata todos los `NaN` como un mismo valor; usa `df.duplicated(subset=['Telefono']) & df['Telefono'].notna()`.
- Suponer que `duplicated().sum() == 0` significa datos limpios: puede haber duplicados ocultos por formato, tipo o identificadores distintos.

---
**Fin del notebook.**